# 6. The repository's own verification suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/main/notebooks/06_verification_suite.ipynb)

Runs `python verify.py` (the fast path, about 6 minutes plus about 1 minute of installation): table and prose recomputation, regression tests, certificates and the file manifest. The core verifier also regenerates the figures and compares them byte for byte, which needs the exact library versions of `requirements-lock.txt`. The notebook therefore runs the suite inside a small dedicated environment with those versions, leaving your own Python installation unchanged. The longer suites are `python verify.py --full` and `--everything`; the Lean proofs need a Lean toolchain (see `LEAN_VERIFICATION.md`).

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the repository first).
import os, sys, subprocess
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "scikit-learn"], check=True)
ROOT = os.getcwd()
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

working directory: /Users/ganeshshiwakoti/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

In [3]:
import venv
env = os.path.join(ROOT, ".venv-verify")
py = os.path.join(env, "bin", "python")
if not os.path.exists(py):
    venv.create(env, with_pip=True)
    run(f"{py} -m pip install -q -r requirements-lock.txt", tail=3)
run(f"{py} -m pip list --format=freeze", tail=20)

contourpy==1.3.3
cycler==0.12.1
fonttools==4.62.1
kiwisolver==1.5.0
matplotlib==3.10.8
mpmath==1.3.0
networkx==3.6.1
numpy==2.4.4
packaging==26.0
pillow==12.2.0
pip==24.2
pyparsing==3.3.2
python-dateutil==2.9.0.post0
scipy==1.17.1
six==1.17.0
sympy==1.14.0
[exit 0, 0 s]


'contourpy==1.3.3\ncycler==0.12.1\nfonttools==4.62.1\nkiwisolver==1.5.0\nmatplotlib==3.10.8\nmpmath==1.3.0\nnetworkx==3.6.1\nnumpy==2.4.4\npackaging==26.0\npillow==12.2.0\npip==24.2\npyparsing==3.3.2\npython-dateutil==2.9.0.post0\nscipy==1.17.1\nsix==1.17.0\nsympy==1.14.0\n'

In [4]:
out = run(f"{py} verify.py", tail=40)
check("fast path result", True, "ALL PASS" in out)

[PASS] core verifier (144 s): scope: FULL PASS covers the executed computations, all 8 numeric tables of main.tex/supplement.tex (exact display strings), and canonical ar
[PASS] prose numbers (5 s): prose numbers: 705 checked, 0 failures, 18 uncheckable (cited values; see UNCHECKABLE in the registries)
[PASS] external regression tests (1 s): OK (expected failures=1)
[PASS] baseline tests (54 s): 0 failures; 54.4s
[PASS] constancy test check (7 s): RESULT: the two constancy tests never disagree on these inputs; no fold is affected
[PASS] auxiliary outputs (6 s): Auxiliary output reproduction PASS
[PASS] archived-source identities (0 s): Archived source formula/skeleton checks PASS: 154 pages
[PASS] interval certificates (1 s): Mathematics interval checks PASS
[PASS] prose style (0 s): prose style check: PASS
[PASS] extremal trees (0 s): RESULT: extremal-tree proposition holds for all orders 4..12
[PASS] collision minimality (3 s): RESULT: first order with distinct profiles sharing a q-h

True

## Summary

In [5]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                                                                                             check   paper recomputed status
0  script: /Users/ganeshshiwakoti/LOYOLA-PAPER/.venv-verify/bin/python -m pip list --format=freeze  exit 0     exit 0     OK
1                    script: /Users/ganeshshiwakoti/LOYOLA-PAPER/.venv-verify/bin/python verify.py  exit 0     exit 0     OK
2                                                                                 fast path result    True       True     OK
3 checks, 0 mismatches
